# TP Jour 4 — Tests d'Hypothèses Statistiques, Segmentation & Restitution BI

### Objectifs de la séance
1. Valider scientifiquement deux hypothèses métier sur votre dataset à l'aide de **tests statistiques adaptés** (`scipy.stats`).
2. Construire une **segmentation / typologie non supervisée (K-Means)** combinant agrégation distribuée **PySpark**, transformation d'asymétrie (`log1p`), standardisation (`StandardScaler`) et labellisation métier vers **`/lakehouse/gold/`**.
3. Développer votre **Dashboard Streamlit interactif** dans `app/dashboard.py` (`http://localhost:8501`) en documentant chaque requête SQL et chaque choix statistique.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J4_Stats_Segmentation_Dashboard")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-17c37551-2032-465f-91bd-fed0f4766f50;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

## 1. Inférence Statistique : Tester vos Hypothèses Métier (`scipy.stats`)

Choisissez **au moins deux tests statistiques** pertinents pour votre sujet parmi :
- **Comparaison de 2 groupes indépendants** sur une variable asymétrique : **Mann-Whitney U** (`stats.mannwhitneyu`).
- **Comparaison de $k \ge 3$ groupes** : **Kruskal-Wallis** (`stats.kruskal`).
- **Lien monotone entre 2 variables quantitatives** non gaussiennes : **Corrélation de rangs de Spearman** (`stats.spearmanr`).
- **Dépendance entre 2 variables qualitatives** : **Test du $\chi^2$** (`stats.chi2_contingency`).

### Exercice 1.1 — Formuler $H_0 / H_1$, projeter les colonnes utiles depuis Silver et conclure
> **Bonne pratique Big Data :** Ne faites jamais `.toPandas()` sur toute une table Silver brute ! Filtrez et sélectionnez (`select(...)`) uniquement les colonnes nécessaires côté PySpark avant de rapatrier les données vers le driver.

**Formulation des hypothèses**
- **Mann–Whitney U**. H₀ : les médianes d’indemnités par club ont la même distribution selon que le club recrute majoritairement entre championnats (≥ 50 %) ou sous ce seuil. H₁ : les distributions diffèrent.
- **Spearman**. H₀ : il n’y a pas de corrélation de rang entre le nombre d’arrivées et l’indemnité médiane du club. H₁ : la corrélation de rang diffère de zéro.
J’applique la correction de Bonferroni parce que je teste deux hypothèses. J’interprète les résultats comme des associations exploratoires, car les clubs peuvent partager les mêmes marchés de recrutement.

In [2]:
from pyspark.sql import functions as F
GOLD = "/lakehouse/gold"
# Données de faits depuis Silver, affiliations historiques depuis Bronze.
silver = spark.read.format("delta").load("/lakehouse/silver/football_transfers").select(
    F.col("year").try_cast("int").alias("year"),
    F.col("left_club_id").try_cast("bigint").alias("origin_club_id"),
    F.col("joined_club_id").try_cast("bigint").alias("destination_club_id"),
    F.col("club_name").alias("destination_club"),
    F.when((F.col("transfer_fee") >= 0) & (F.col("transfer_fee") < float("inf")),
           F.col("transfer_fee")).alias("fee"),
)
memberships = spark.read.format("delta").load("/lakehouse/bronze/club_league").select(
    "club_id", "league_id", "season"
)
season = F.trim(F.col("season").cast("string"))
start = F.regexp_extract(season, r"^((?:19|20)\d{2})$", 1)
span = r"^((?:19|20)\d{2})\s*[-/]\s*((?:19|20)\d{2}|\d{2})$"
span_start = F.regexp_extract(season, span, 1)
span_end = F.regexp_extract(season, span, 2)
year_start = F.when(start != "", start).otherwise(span_start).try_cast("int")
short_end = (F.floor(year_start / 100) * 100 + span_end.try_cast("int"))
year_end = (
    F.when(start != "", year_start)
    .when(F.length(span_end) == 2, short_end + F.when(short_end < year_start, 100).otherwise(0))
    .otherwise(span_end.try_cast("int"))
)

parsed = memberships.select(
    F.col("club_id").try_cast("bigint").alias("club_id"),
    F.nullif(F.trim(F.col("league_id").cast("string")), F.lit("")).alias("league_id"),
    year_start.alias("year_start"),
    year_end.cast("int").alias("year_end"),
)
valid = F.col("year_start").isNotNull() & F.col("year_end").between(
    F.col("year_start"), F.col("year_start") + 1
)
lookup = (
    parsed.where(valid & F.col("club_id").isNotNull() & F.col("league_id").isNotNull())
    .withColumn("year", F.explode(F.sequence("year_start", "year_end")))
    .groupBy("club_id", "year")
    .agg(F.countDistinct("league_id").alias("league_count"), F.min("league_id").alias("candidate"))
    .withColumn("league_id", F.when(F.col("league_count") == 1, F.col("candidate")))
    .drop("candidate")
)
data = silver
for side in ("origin", "destination"):
    club_key = f"{side}_club_id"
    affiliations = lookup.select(
        "year", F.col("club_id").alias(club_key), F.col("league_id").alias(f"{side}_league_id")
    )
    data = data.join(affiliations, ["year", club_key], "left")

assert data.count() == silver.count(), "La jointure a modifié le nombre de transferts."

data.createOrReplaceTempView("football_transfers_enriched")
last_year = spark.sql("""
    SELECT MAX(year) AS year FROM football_transfers_enriched
    WHERE origin_league_id IS NOT NULL AND destination_league_id IS NOT NULL
""").first()["year"]
assert last_year is not None, "Aucune année rattachée. Vérifier les formats de saison en J3."
first_year = int(last_year) - 2
entities = spark.sql("""
SELECT destination_club_id AS club_id, MIN(destination_club) AS club,
       COUNT(*) AS arrivals, COUNT(DISTINCT origin_club_id) AS origin_clubs,
       COUNT_IF(fee > 0) AS known_positive_fees,
       PERCENTILE_APPROX(IF(fee > 0, fee, NULL), 0.50) AS median_fee,
       AVG(CAST(origin_league_id <> destination_league_id AS DOUBLE)) AS cross_share
FROM football_transfers_enriched
WHERE year BETWEEN {first_year} AND {last_year}
  AND origin_league_id IS NOT NULL AND destination_league_id IS NOT NULL
  AND destination_club_id IS NOT NULL AND destination_club IS NOT NULL
GROUP BY destination_club_id
HAVING COUNT(*) >= 10 AND COUNT_IF(fee > 0) >= 5
""".format(first_year=first_year, last_year=int(last_year))).orderBy("club_id").toPandas()
assert len(entities) >= 3, "Moins de trois clubs éligibles. Examiner la couverture J3."
print(f"Fenêtre {first_year}–{last_year} : {len(entities)} clubs éligibles.")

cross = entities.loc[entities["cross_share"] >= 0.5, "median_fee"]
other = entities.loc[entities["cross_share"] < 0.5, "median_fee"]
alpha = 0.05 / 2

def report(label, result):
    decision = "H₀ rejetée" if result.pvalue < alpha else "H₀ non rejetée"
    print(f"{label} : statistique={result.statistic:.4g}, p={result.pvalue:.4g}, {decision} (seuil {alpha}).")

if min(len(cross), len(other)) >= 5:
    print(f"Clubs par groupe : {len(cross)} et {len(other)}.")
    report("Mann–Whitney U", stats.mannwhitneyu(cross, other, alternative="two-sided"))
else:
    print("Mann–Whitney U : groupes trop petits (moins de cinq clubs dans un groupe).")

if entities["arrivals"].nunique() > 1 and entities["median_fee"].nunique() > 1:
    report("Spearman", stats.spearmanr(entities["arrivals"], entities["median_fee"]))
else:
    print("Spearman : variable constante.")

26/11/18 16:31:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Fenêtre 2022–2024 : 419 clubs éligibles.
Clubs par groupe : 388 et 31.
Mann–Whitney U : statistique=5464, p=0.3968, H₀ non rejetée (seuil 0.025).
Spearman : statistique=-0.2152, p=8.833e-06, H₀ rejetée (seuil 0.025).


## 2. Segmentation / Typologie par Clustering K-Means (`/lakehouse/gold/`)

Quelle que soit la thématique de votre dataset Kaggle, identifiez l'**entité métier pertinente à segmenter** (ex. clients/utilisateurs, stations, zones géographiques, produits, lignes de transport, capteurs...) :
1. **Agrégation PySpark** : résumez chaque entité par **3 à 5 features numériques comportementales** (fréquence, volume, valeur médiane/totale, récence, taux d'incident...).
2. **Prétraitement statistique** : appliquez `np.log1p(...)` sur les features positives asymétriques puis centrez-réduisez avec `StandardScaler()` afin qu'aucune variable à grande échelle n'écrase la distance euclidienne du K-Means.
3. **Interprétation métier** : inspectez les médianes par cluster (`groupby("cluster_id").median()`), attribuez un **nom explicite** à chaque segment (`colonne "segment"`), puis enregistrez la table dans **`/lakehouse/gold/<nom_table_segments>`**.

In [3]:
features = ["arrivals", "origin_clubs", "median_fee", "cross_share"]
values = entities[features].astype(float).copy()
# Je réduis l’effet des gros volumes et des grosses indemnités avant de mettre les variables à la même échelle.
values[features[:3]] = np.log1p(values[features[:3]])
assert len(values.drop_duplicates()) >= 3, "Moins de trois profils numériques distincts."
scaled = StandardScaler().fit_transform(values)
entities["cluster_id"] = KMeans(n_clusters=3, n_init=20, random_state=42).fit_predict(scaled).astype("int32")
profiles = entities.groupby("cluster_id")[features].median()
reference = entities[features].median()
labels = {}
for cluster_id, profile in profiles.iterrows():
    activity = "activité élevée" if profile["arrivals"] >= reference["arrivals"] else "activité modérée"
    fee = "indemnité élevée" if profile["median_fee"] >= reference["median_fee"] else "indemnité modérée"
    cross_label = "recrutement externe" if profile["cross_share"] >= 0.5 else "recrutement interne"
    labels[cluster_id] = f"Profil {cluster_id + 1} · {activity}, {fee}, {cross_label}"
entities["segment"] = entities["cluster_id"].map(labels)
entities["first_year"] = int(first_year)
entities["last_year"] = int(last_year)
display(profiles.assign(segment=profiles.index.map(labels)))
(
    spark.createDataFrame(entities).coalesce(2).write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/football_league_segments")
)

,arrivals,origin_clubs,median_fee,cross_share,segment
cluster_id,,,,,
0,31.0,24.0,3000000.0,0.837838,"Profil 1 · activité modérée, indemnité élevée,..."
1,24.0,18.0,1000000.0,0.625000,"Profil 2 · activité modérée, indemnité élevée,..."
2,41.0,31.0,371000.0,0.800000,"Profil 3 · activité élevée, indemnité modérée,..."


/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/opt/venv/lib/python3.15/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## 3. Livrable Final : Votre Application `app/dashboard.py` sur Streamlit (`:8501`)

Complétez maintenant **`app/dashboard.py`** pour valoriser vos tables `/lakehouse/gold/` :
- **Architecture respectée** : l'application Streamlit ne doit lire **que** des tables pré-calculées de la couche **Gold** (`/lakehouse/gold/...`), jamais les fichiers CSV bruts !
- **Interactivité** : au moins un filtre utilisateur (`st.selectbox`, `st.multiselect`, `st.slider`) et 3 à 4 visualisations Plotly Express complémentaires.
- **Rigueur explicative** : sous chaque visualisation clé, intégrez un `st.expander("Requête SQL & Justification statistique")` affichant la requête Spark SQL ayant produit la table Gold et justifiant le choix des indicateurs statistiques retenus.

In [4]:
# Vérification finale de l'inventaire complet de votre couche Gold sur HDFS
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
PathHDFS = jvm.org.apache.hadoop.fs.Path
gold_root = PathHDFS("/lakehouse/gold")

gold_tables = [
    s.getPath().getName()
    for s in fs.listStatus(gold_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(gold_tables) > 0, "Aucune table Delta trouvée dans /lakehouse/gold/ !"

print("=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===")
for t in sorted(gold_tables):
    cnt = spark.read.format("delta").load(f"/lakehouse/gold/{t}").count()
    print(f" - /lakehouse/gold/{t:<25} : {cnt:>9,} lignes")

=== Tables Delta disponibles dans /lakehouse/gold/ pour Streamlit ===


 - /lakehouse/gold/football_league_annual    :       219 lignes


 - /lakehouse/gold/football_league_coverage  :       691 lignes


 - /lakehouse/gold/football_league_enriched  : 3,469,314 lignes


 - /lakehouse/gold/football_league_flows     :   103,293 lignes
 - /lakehouse/gold/football_league_pareto    :     7,819 lignes
 - /lakehouse/gold/football_league_segments  :       419 lignes
 - /lakehouse/gold/football_league_transfers :   757,798 lignes
